<a href="https://www.kaggle.com/code/jf10101001/group04-bloodmnist-task2-baselines?scriptVersionId=342323560" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# CSE475 - Section 02

- - -

**Group Information - Group 04 (BloodMNIST)**

* Nishat Subha Mithela (2023-1-60-248) - EDA

* Nusrat Jahan (2022-2-60-128) - Baselines

* Humayara Nahar Moumita (2021-2-60-121) - CNN + Attention

* Fahmida Jannat (2025-1-60-334) - GNN





- - -


**Course Instructor**

Dr. Raihan Ul Islam

Associate Professor

Department of Computer Science & Engineering

East West University

# Dataset Load

In [ ]:
# =========================================================
# DATASET LOAD
# =========================================================

# Install only the packages required for loading the dataset
%pip -q install -U kagglehub

In [ ]:
# =========================================================
# 1. Imports and reproducibility
# =========================================================

from pathlib import Path
import random

import kagglehub
import numpy as np
import tensorflow as tf


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# =========================================================
# 2. Download BloodMNIST
# =========================================================

DATA_DIR = Path(
    kagglehub.dataset_download(
        "subbhaa/bloodmnist-medmnist-dataset"
    )
)

print("Dataset directory:", DATA_DIR)

In [ ]:
# =========================================================
# 3. Find and load the BloodMNIST NPZ file
# =========================================================

npz_files = sorted(
    DATA_DIR.rglob("*.npz")
)

assert npz_files, (
    "No NPZ dataset file was found."
)

NPZ_PATH = next(
    (
        path
        for path in npz_files
        if path.name.lower() == "bloodmnist.npz"
    ),
    npz_files[0]
)

with np.load(
    NPZ_PATH,
    allow_pickle=False
) as data:

    required_arrays = [
        "train_images",
        "train_labels",
        "val_images",
        "val_labels",
        "test_images",
        "test_labels"
    ]

    missing_arrays = [
        name
        for name in required_arrays
        if name not in data.files
    ]

    assert not missing_arrays, (
        f"Missing arrays: {missing_arrays}"
    )

    # Copy arrays so they remain available after closing NPZ
    X_train = data["train_images"].copy()
    y_train = data["train_labels"].reshape(-1).copy()

    X_val = data["val_images"].copy()
    y_val = data["val_labels"].reshape(-1).copy()

    X_test = data["test_images"].copy()
    y_test = data["test_labels"].reshape(-1).copy()


# Ensure correct label data type
y_train = y_train.astype(np.int32)
y_val = y_val.astype(np.int32)
y_test = y_test.astype(np.int32)


print("Loaded:", NPZ_PATH.name)

In [ ]:
# =========================================================
# 4. Class information
# =========================================================

CLASS_NAMES = {
    0: "Basophil",
    1: "Eosinophil",
    2: "Erythroblast",
    3: "Immature granulocytes",
    4: "Lymphocyte",
    5: "Monocyte",
    6: "Neutrophil",
    7: "Platelet"
}

class_names = [
    CLASS_NAMES[class_id]
    for class_id in sorted(CLASS_NAMES)
]

NUM_CLASSES = len(class_names)

In [ ]:
# =========================================================
# 5. Validate the loaded dataset
# =========================================================

raw_splits = {
    "Train": (X_train, y_train),
    "Validation": (X_val, y_val),
    "Test": (X_test, y_test)
}

for split_name, (images, labels) in raw_splits.items():

    assert isinstance(images, np.ndarray), (
        f"{split_name}: images must be a NumPy array."
    )

    assert isinstance(labels, np.ndarray), (
        f"{split_name}: labels must be a NumPy array."
    )

    assert images.ndim == 4, (
        f"{split_name}: expected image shape "
        f"(samples, height, width, channels), "
        f"but received {images.shape}."
    )

    assert labels.ndim == 1, (
        f"{split_name}: labels must be one-dimensional."
    )

    assert len(images) == len(labels), (
        f"{split_name}: number of images and labels differs."
    )

    assert labels.min() >= 0, (
        f"{split_name}: negative class label detected."
    )

    assert labels.max() < NUM_CLASSES, (
        f"{split_name}: invalid class label detected."
    )


print("Dataset loading and validation completed.")
print("-" * 60)

print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)

print("Validation images:", X_val.shape)
print("Validation labels:", y_val.shape)

print("Test images:", X_test.shape)
print("Test labels:", y_test.shape)

print("\nClasses:", NUM_CLASSES)
print("Class names:", class_names)

# Dataset EDA

In [ ]:
# =========================================================
# OPTIONAL EXPLORATORY DATA ANALYSIS
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf


# =========================================================
# 1. Verify that the Dataset Load section was executed
# =========================================================

eda_required_variables = [
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_test",
    "y_test",
    "CLASS_NAMES"
]

eda_missing_variables = [
    variable_name
    for variable_name in eda_required_variables
    if variable_name not in globals()
]

if eda_missing_variables:
    raise RuntimeError(
        "Run the Dataset Load section before running EDA. "
        f"Missing variables: {eda_missing_variables}"
    )


# Create class information locally for EDA
if isinstance(CLASS_NAMES, dict):
    eda_class_names = [
        CLASS_NAMES[class_id]
        for class_id in sorted(CLASS_NAMES)
    ]
else:
    eda_class_names = list(CLASS_NAMES)

eda_num_classes = len(eda_class_names)


# Local split dictionary used only by EDA
eda_splits = {
    "Train": (
        np.asarray(X_train),
        np.asarray(y_train).reshape(-1)
    ),
    "Validation": (
        np.asarray(X_val),
        np.asarray(y_val).reshape(-1)
    ),
    "Test": (
        np.asarray(X_test),
        np.asarray(y_test).reshape(-1)
    )
}


print("EDA dependencies are ready.")
print("Number of classes:", eda_num_classes)
print("Class names:", eda_class_names)

In [ ]:
# =========================================================
# 2. Dataset structure summary
# =========================================================

eda_structure_rows = []

for split_name, (images, labels) in eda_splits.items():

    eda_structure_rows.append({
        "Split": split_name,
        "Number of Images": len(images),
        "Image Shape": str(images.shape[1:]),
        "Images Dtype": str(images.dtype),
        "Labels Shape": str(labels.shape),
        "Labels Dtype": str(labels.dtype),
        "Minimum Label": int(labels.min()),
        "Maximum Label": int(labels.max())
    })


eda_structure_df = pd.DataFrame(
    eda_structure_rows
)

print("Dataset structure")
display(eda_structure_df)

In [ ]:
# =========================================================
# 3. Dataset split sizes
# =========================================================

eda_split_sizes = {
    split_name: len(images)
    for split_name, (images, _) in eda_splits.items()
}

eda_total_samples = sum(
    eda_split_sizes.values()
)

eda_split_df = pd.DataFrame({
    "Split": list(eda_split_sizes.keys()),
    "Samples": list(eda_split_sizes.values())
})

eda_split_df["Percentage"] = (
    eda_split_df["Samples"]
    / eda_total_samples
    * 100
)

print("Dataset split summary")
display(eda_split_df)

In [ ]:
# =========================================================
# 4. Class distribution for each split
# =========================================================

eda_distribution_rows = []

for split_name, (_, labels) in eda_splits.items():

    split_counts = np.bincount(
        labels.astype(np.int32),
        minlength=eda_num_classes
    )

    split_total = len(labels)

    for class_id in range(eda_num_classes):

        eda_distribution_rows.append({
            "Split": split_name,
            "Class ID": class_id,
            "Class Name": eda_class_names[class_id],
            "Count": int(split_counts[class_id]),
            "Percentage": (
                split_counts[class_id]
                / split_total
                * 100
            )
        })


eda_distribution_df = pd.DataFrame(
    eda_distribution_rows
)

print("Class distribution across all splits")
display(eda_distribution_df)

In [ ]:
# =========================================================
# 5. Class-count comparison table
# =========================================================

eda_class_count_table = (
    eda_distribution_df
    .pivot(
        index=[
            "Class ID",
            "Class Name"
        ],
        columns="Split",
        values="Count"
    )
    .reset_index()
)

eda_class_count_table.columns.name = None

expected_columns = [
    "Class ID",
    "Class Name",
    "Train",
    "Validation",
    "Test"
]

eda_class_count_table = eda_class_count_table[
    expected_columns
]

eda_class_count_table["Total"] = (
    eda_class_count_table["Train"]
    + eda_class_count_table["Validation"]
    + eda_class_count_table["Test"]
)

print("Class-count comparison")
display(eda_class_count_table)

In [ ]:
# =========================================================
# 6. Plot class distribution for each split
# =========================================================

for split_name, (_, labels) in eda_splits.items():

    split_counts = np.bincount(
        labels.astype(np.int32),
        minlength=eda_num_classes
    )

    plt.figure(figsize=(12, 5))

    plt.bar(
        eda_class_names,
        split_counts
    )

    plt.title(
        f"{split_name} Class Distribution"
    )

    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(
        rotation=45,
        ha="right"
    )

    for index, count in enumerate(split_counts):
        plt.text(
            index,
            count,
            str(int(count)),
            ha="center",
            va="bottom",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

In [ ]:
# =========================================================
# 7. Training-class imbalance analysis
# =========================================================

eda_train_counts = np.bincount(
    np.asarray(y_train).reshape(-1).astype(np.int32),
    minlength=eda_num_classes
)

eda_min_class_id = int(
    np.argmin(eda_train_counts)
)

eda_max_class_id = int(
    np.argmax(eda_train_counts)
)

eda_min_count = int(
    eda_train_counts[eda_min_class_id]
)

eda_max_count = int(
    eda_train_counts[eda_max_class_id]
)

eda_imbalance_ratio = (
    eda_max_count / eda_min_count
)


eda_imbalance_df = pd.DataFrame({
    "Class ID": np.arange(eda_num_classes),
    "Class Name": eda_class_names,
    "Training Count": eda_train_counts,
    "Difference from Minimum": (
        eda_train_counts - eda_min_count
    ),
    "Ratio to Minimum": (
        eda_train_counts / eda_min_count
    )
})

print("Training imbalance analysis")
display(eda_imbalance_df)


print("\nTraining imbalance summary")
print("-" * 60)

print(
    f"Smallest class: "
    f"{eda_class_names[eda_min_class_id]} "
    f"({eda_min_count} samples)"
)

print(
    f"Largest class: "
    f"{eda_class_names[eda_max_class_id]} "
    f"({eda_max_count} samples)"
)

print(
    f"Maximum-to-minimum ratio: "
    f"{eda_imbalance_ratio:.3f}:1"
)

In [ ]:
# =========================================================
# 8. Image-value statistics
# =========================================================

eda_pixel_statistics = []

for split_name, (images, _) in eda_splits.items():

    images_float = images.astype(
        np.float32,
        copy=False
    )

    eda_pixel_statistics.append({
        "Split": split_name,
        "Minimum Pixel": float(
            np.min(images_float)
        ),
        "Maximum Pixel": float(
            np.max(images_float)
        ),
        "Mean Pixel": float(
            np.mean(images_float)
        ),
        "Standard Deviation": float(
            np.std(images_float)
        )
    })


eda_pixel_stats_df = pd.DataFrame(
    eda_pixel_statistics
)

print("Pixel-value statistics")
display(eda_pixel_stats_df)

In [ ]:
# =========================================================
# 9. Per-channel RGB statistics
# =========================================================

eda_channel_names = [
    "Red",
    "Green",
    "Blue"
]

eda_channel_rows = []

for split_name, (images, _) in eda_splits.items():

    if images.ndim != 4:
        continue

    number_of_channels = images.shape[-1]

    for channel_index in range(number_of_channels):

        channel_values = images[
            ...,
            channel_index
        ].astype(
            np.float32,
            copy=False
        )

        channel_name = (
            eda_channel_names[channel_index]
            if channel_index < len(eda_channel_names)
            else f"Channel {channel_index}"
        )

        eda_channel_rows.append({
            "Split": split_name,
            "Channel": channel_name,
            "Mean": float(
                np.mean(channel_values)
            ),
            "Standard Deviation": float(
                np.std(channel_values)
            ),
            "Minimum": float(
                np.min(channel_values)
            ),
            "Maximum": float(
                np.max(channel_values)
            )
        })


eda_channel_stats_df = pd.DataFrame(
    eda_channel_rows
)

print("Per-channel image statistics")
display(eda_channel_stats_df)

In [ ]:
# =========================================================
# 10. Random training samples
# =========================================================

EDA_RANDOM_SEED = 42
EDA_NUMBER_OF_SAMPLES = 16

eda_rng = np.random.default_rng(
    EDA_RANDOM_SEED
)

eda_sample_count = min(
    EDA_NUMBER_OF_SAMPLES,
    len(X_train)
)

eda_random_indices = eda_rng.choice(
    len(X_train),
    size=eda_sample_count,
    replace=False
)


eda_columns = 4
eda_rows = int(
    np.ceil(
        eda_sample_count / eda_columns
    )
)

plt.figure(
    figsize=(12, 3 * eda_rows)
)

for plot_position, sample_index in enumerate(
    eda_random_indices,
    start=1
):

    image = X_train[sample_index]
    label_id = int(y_train[sample_index])

    plt.subplot(
        eda_rows,
        eda_columns,
        plot_position
    )

    plt.imshow(image)

    plt.title(
        f"{eda_class_names[label_id]}\n"
        f"Class ID: {label_id}"
    )

    plt.axis("off")


plt.suptitle(
    "Random BloodMNIST Training Images",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 11. One training image from each class
# =========================================================

plt.figure(
    figsize=(16, 8)
)

for class_id in range(eda_num_classes):

    class_indices = np.flatnonzero(
        np.asarray(y_train).reshape(-1)
        == class_id
    )

    if len(class_indices) == 0:
        continue

    sample_index = class_indices[0]

    plt.subplot(
        2,
        4,
        class_id + 1
    )

    plt.imshow(
        X_train[sample_index]
    )

    plt.title(
        f"{eda_class_names[class_id]}\n"
        f"Class ID: {class_id}"
    )

    plt.axis("off")


plt.suptitle(
    "One Training Sample from Each Class",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 13. Mean image brightness by split
# =========================================================

plt.figure(
    figsize=(10, 6)
)

for split_name, (images, _) in eda_splits.items():

    image_brightness = np.mean(
        images.astype(
            np.float32,
            copy=False
        ),
        axis=(1, 2, 3)
    )

    plt.hist(
        image_brightness,
        bins=40,
        alpha=0.5,
        label=split_name
    )


plt.title(
    "Distribution of Mean Image Brightness"
)

plt.xlabel(
    "Mean Pixel Intensity per Image"
)

plt.ylabel(
    "Number of Images"
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 14. Mean image brightness by training class
# =========================================================

eda_train_images_float = X_train.astype(
    np.float32,
    copy=False
)

eda_train_brightness = np.mean(
    eda_train_images_float,
    axis=(1, 2, 3)
)

eda_brightness_rows = []

for class_id in range(eda_num_classes):

    class_mask = (
        np.asarray(y_train).reshape(-1)
        == class_id
    )

    class_brightness = eda_train_brightness[
        class_mask
    ]

    eda_brightness_rows.append({
        "Class ID": class_id,
        "Class Name": eda_class_names[class_id],
        "Mean Brightness": float(
            np.mean(class_brightness)
        ),
        "Brightness Std": float(
            np.std(class_brightness)
        ),
        "Minimum Brightness": float(
            np.min(class_brightness)
        ),
        "Maximum Brightness": float(
            np.max(class_brightness)
        )
    })


eda_brightness_df = pd.DataFrame(
    eda_brightness_rows
)

print("Training-image brightness by class")
display(eda_brightness_df)

In [ ]:
plt.figure(
    figsize=(12, 5)
)

plt.bar(
    eda_brightness_df["Class Name"],
    eda_brightness_df["Mean Brightness"]
)

plt.title(
    "Average Training-Image Brightness by Class"
)

plt.xlabel("Class")
plt.ylabel("Mean Pixel Intensity")

plt.xticks(
    rotation=45,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 15. Mean RGB values by training class
# =========================================================

eda_rgb_rows = []

for class_id in range(eda_num_classes):

    class_images = X_train[
        np.asarray(y_train).reshape(-1)
        == class_id
    ].astype(
        np.float32,
        copy=False
    )

    channel_means = np.mean(
        class_images,
        axis=(0, 1, 2)
    )

    eda_rgb_rows.append({
        "Class ID": class_id,
        "Class Name": eda_class_names[class_id],
        "Red Mean": float(channel_means[0]),
        "Green Mean": float(channel_means[1]),
        "Blue Mean": float(channel_means[2])
    })


eda_rgb_df = pd.DataFrame(
    eda_rgb_rows
)

print("Mean RGB values by training class")
display(eda_rgb_df)

In [ ]:
# =========================================================
# 16. Data-integrity checks
# =========================================================

eda_integrity_rows = []

for split_name, (images, labels) in eda_splits.items():

    invalid_low_labels = int(
        np.sum(labels < 0)
    )

    invalid_high_labels = int(
        np.sum(labels >= eda_num_classes)
    )

    nan_image_values = int(
        np.isnan(
            images.astype(
                np.float32,
                copy=False
            )
        ).sum()
    )

    infinite_image_values = int(
        np.isinf(
            images.astype(
                np.float32,
                copy=False
            )
        ).sum()
    )

    eda_integrity_rows.append({
        "Split": split_name,
        "Invalid Negative Labels": invalid_low_labels,
        "Labels Above Class Range": invalid_high_labels,
        "NaN Pixel Values": nan_image_values,
        "Infinite Pixel Values": infinite_image_values
    })


eda_integrity_df = pd.DataFrame(
    eda_integrity_rows
)

print("Data-integrity checks")
display(eda_integrity_df)

In [ ]:
# =========================================================
# 17. Exact duplicates within each split
# =========================================================

import hashlib


def eda_image_hash(image):
    """
    Generate a deterministic hash for one image.
    """

    contiguous_image = np.ascontiguousarray(
        image
    )

    return hashlib.sha256(
        contiguous_image.tobytes()
    ).hexdigest()


eda_duplicate_rows = []

for split_name, (images, _) in eda_splits.items():

    split_hashes = [
        eda_image_hash(image)
        for image in images
    ]

    unique_hash_count = len(
        set(split_hashes)
    )

    duplicate_count = (
        len(split_hashes)
        - unique_hash_count
    )

    eda_duplicate_rows.append({
        "Split": split_name,
        "Total Images": len(images),
        "Unique Images": unique_hash_count,
        "Exact Duplicate Images": duplicate_count,
        "Duplicate Percentage": (
            duplicate_count
            / len(images)
            * 100
        )
    })


eda_duplicate_df = pd.DataFrame(
    eda_duplicate_rows
)

print("Exact duplicate images within each split")
display(eda_duplicate_df)

In [ ]:
# =========================================================
# 18. Exact image overlap between dataset splits
# =========================================================

eda_split_hashes = {}

for split_name, (images, _) in eda_splits.items():

    eda_split_hashes[split_name] = {
        eda_image_hash(image)
        for image in images
    }


eda_overlap_pairs = [
    ("Train", "Validation"),
    ("Train", "Test"),
    ("Validation", "Test")
]

eda_overlap_rows = []

for first_split, second_split in eda_overlap_pairs:

    overlap_count = len(
        eda_split_hashes[first_split]
        .intersection(
            eda_split_hashes[second_split]
        )
    )

    eda_overlap_rows.append({
        "First Split": first_split,
        "Second Split": second_split,
        "Exact Image Overlap": overlap_count
    })


eda_overlap_df = pd.DataFrame(
    eda_overlap_rows
)

print("Exact image overlap between splits")
display(eda_overlap_df)

In [ ]:
# =========================================================
# 19. Final EDA summary
# =========================================================

print("=" * 70)
print("BLOODMNIST EDA SUMMARY")
print("=" * 70)

print(
    f"Total dataset samples: "
    f"{eda_total_samples}"
)

print(
    f"Training samples: "
    f"{len(X_train)}"
)

print(
    f"Validation samples: "
    f"{len(X_val)}"
)

print(
    f"Test samples: "
    f"{len(X_test)}"
)

print(
    f"\nOriginal image shape: "
    f"{X_train.shape[1:]}"
)

print(
    f"Number of classes: "
    f"{eda_num_classes}"
)

print(
    f"\nSmallest training class: "
    f"{eda_class_names[eda_min_class_id]} "
    f"({eda_min_count} samples)"
)

print(
    f"Largest training class: "
    f"{eda_class_names[eda_max_class_id]} "
    f"({eda_max_count} samples)"
)

print(
    f"Training imbalance ratio: "
    f"{eda_imbalance_ratio:.3f}:1"
)

print(
    f"\nTraining pixel range: "
    f"{np.min(X_train)} to {np.max(X_train)}"
)

print(
    f"Training image dtype: "
    f"{X_train.dtype}"
)

print("=" * 70)

# Dataset Preprocessing

# Processing configuration

In [ ]:
# =========================================================
# DATASET PROCESSING
# =========================================================

import numpy as np
import pandas as pd
import tensorflow as tf


# =========================================================
# 1. Processing configuration
# =========================================================

SEED = 42

IMG_HEIGHT = 224
IMG_WIDTH = 224

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# The threshold is calculated near the minority-class counts.
#
# threshold =
# median of lower-half class counts × this ratio
MINORITY_THRESHOLD_RATIO = 1.20


np.random.seed(SEED)
tf.random.set_seed(SEED)

sampling_rng = np.random.default_rng(SEED)

Validate load-section variables

In [ ]:
# =========================================================
# DATASET PROCESSING
# =========================================================

import numpy as np
import pandas as pd
import tensorflow as tf


# =========================================================
# 1. Processing configuration
# =========================================================

SEED = 42

IMG_HEIGHT = 224
IMG_WIDTH = 224

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# The threshold is calculated near the minority-class counts.
#
# threshold =
# median of lower-half class counts × this ratio
MINORITY_THRESHOLD_RATIO = 1.20


np.random.seed(SEED)
tf.random.set_seed(SEED)

sampling_rng = np.random.default_rng(SEED)

# Calculate the minority-based threshol

In [ ]:
# =========================================================
# 3. Calculate original class distribution
# =========================================================

original_counts = np.bincount(
    y_train,
    minlength=NUM_CLASSES
)

original_distribution = pd.DataFrame({
    "Class ID": np.arange(NUM_CLASSES),
    "Class Name": class_names,
    "Original Count": original_counts
})

print("Original training-class distribution")
display(original_distribution)

In [ ]:
# =========================================================
# 4. Calculate minority-based undersampling threshold
# =========================================================

# Sort counts from smallest to largest
sorted_counts_ascending = np.sort(
    original_counts
)

# Use the lower half of the classes as the
# minority reference group
number_of_minority_classes = int(
    np.ceil(NUM_CLASSES / 2)
)

minority_class_counts = sorted_counts_ascending[
    :number_of_minority_classes
]

# Median provides a robust minority reference
minority_reference_count = float(
    np.median(minority_class_counts)
)

raw_threshold = (
    minority_reference_count
    * MINORITY_THRESHOLD_RATIO
)

# Round to an integer sample count
calculated_threshold = int(
    round(raw_threshold)
)

# Protect every class in the minority reference group
UNDERSAMPLING_THRESHOLD = max(
    calculated_threshold,
    int(minority_class_counts.max())
)


print("\nMinority-based threshold calculation")
print("-" * 65)

print(
    "Counts from smallest to largest:",
    sorted_counts_ascending.tolist()
)

print(
    "Minority reference counts:",
    minority_class_counts.tolist()
)

print(
    f"Minority reference median: "
    f"{minority_reference_count:.2f}"
)

print(
    f"Threshold ratio: "
    f"{MINORITY_THRESHOLD_RATIO:.2f}"
)

print(
    "\nThreshold calculation:"
)

print(
    f"{minority_reference_count:.2f} "
    f"× {MINORITY_THRESHOLD_RATIO:.2f} "
    f"= {raw_threshold:.2f}"
)

print(
    f"Final undersampling threshold: "
    f"{UNDERSAMPLING_THRESHOLD}"
)

print(
    f"\nClasses above "
    f"{UNDERSAMPLING_THRESHOLD} samples "
    "will be undersampled."
)

print(
    "Minority classes will remain unchanged."
)

In [ ]:
# =========================================================
# 5. Undersample classes above the threshold
# =========================================================

selected_indices_by_class = []
sampling_actions = []

for class_id in range(NUM_CLASSES):

    class_indices = np.flatnonzero(
        y_train == class_id
    )

    current_count = len(class_indices)

    if current_count > UNDERSAMPLING_THRESHOLD:

        selected_class_indices = sampling_rng.choice(
            class_indices,
            size=UNDERSAMPLING_THRESHOLD,
            replace=False
        )

        action = "Undersampled"

    else:

        selected_class_indices = class_indices.copy()
        action = "Unchanged"

    selected_indices_by_class.append(
        selected_class_indices
    )

    sampling_actions.append(action)


# Combine class-specific indices
selected_indices = np.concatenate(
    selected_indices_by_class
)

# Shuffle the final training order
sampling_rng.shuffle(
    selected_indices
)


# Create the final training arrays
X_train_prepared = X_train[
    selected_indices
]

y_train_prepared = y_train[
    selected_indices
]

Verify sampling results

In [ ]:
# =========================================================
# 6. Verify final class distribution
# =========================================================

final_counts = np.bincount(
    y_train_prepared,
    minlength=NUM_CLASSES
)

samples_removed = (
    original_counts - final_counts
)

sampling_distribution = pd.DataFrame({
    "Class ID": np.arange(NUM_CLASSES),
    "Class Name": class_names,
    "Original Count": original_counts,
    "Action": sampling_actions,
    "Final Count": final_counts,
    "Samples Removed": samples_removed
})


print("\nFinal training-class distribution")
display(sampling_distribution)


original_imbalance_ratio = (
    original_counts.max()
    / original_counts.min()
)

final_imbalance_ratio = (
    final_counts.max()
    / final_counts.min()
)


print("\nUndersampling result by class")
print("-" * 65)

for class_id, class_name in enumerate(class_names):

    original_count = int(
        original_counts[class_id]
    )

    final_count = int(
        final_counts[class_id]
    )

    removed_count = int(
        samples_removed[class_id]
    )

    if removed_count > 0:
        print(
            f"{class_name}: "
            f"{original_count} - {removed_count} "
            f"= {final_count}"
        )
    else:
        print(
            f"{class_name}: "
            f"{original_count} unchanged"
        )


print("\nFinal processing summary")
print("-" * 65)

print(
    f"Original training samples: "
    f"{len(X_train)}"
)

print(
    f"Processed training samples: "
    f"{len(X_train_prepared)}"
)

print(
    f"Samples removed: "
    f"{len(X_train) - len(X_train_prepared)}"
)

print(
    f"\nOriginal max-to-min ratio: "
    f"{original_imbalance_ratio:.3f}:1"
)

print(
    f"Processed max-to-min ratio: "
    f"{final_imbalance_ratio:.3f}:1"
)

# Image preprocessing

In [ ]:
# =========================================================
# 7. Image preprocessing
# =========================================================

def preprocess_image(image):
    """
    Convert an image to RGB float32 format and resize it.

    Output:
        shape: (224, 224, 3)
        dtype: float32
        range: 0 to 255
    """

    image = tf.convert_to_tensor(image)

    # Ensure a channel dimension exists
    if image.shape.rank == 2:
        image = tf.expand_dims(
            image,
            axis=-1
        )

    image = tf.cast(
        image,
        tf.float32
    )

    # BloodMNIST should already be RGB, but this makes
    # the function safe for grayscale inputs as well.
    if image.shape[-1] == 1:
        image = tf.image.grayscale_to_rgb(
            image
        )

    image = tf.image.resize(
        image,
        size=[IMG_HEIGHT, IMG_WIDTH],
        method=tf.image.ResizeMethod.BILINEAR,
        antialias=True
    )

    image = tf.clip_by_value(
        image,
        clip_value_min=0.0,
        clip_value_max=255.0
    )

    image = tf.ensure_shape(
        image,
        [IMG_HEIGHT, IMG_WIDTH, 3]
    )

    return image

In [ ]:
# =========================================================
# 8. Label preprocessing
# =========================================================

def preprocess_label(label):
    """
    Keep labels as integer class IDs.

    Output shape:
        scalar

    Compatible loss:
        sparse_categorical_crossentropy
    """

    label = tf.cast(
        label,
        tf.int32
    )

    label = tf.ensure_shape(
        label,
        []
    )

    return label

In [ ]:
# =========================================================
# 9. Preprocess one image-label pair
# =========================================================

def preprocess_sample(image, label):

    image = preprocess_image(image)
    label = preprocess_label(label)

    return image, label

# Reusable dataset factory

In [ ]:
# =========================================================
# 10. TensorFlow dataset factory
# =========================================================

def prepare_dataset(
    images,
    labels,
    batch_size=BATCH_SIZE,
    training=False
):
    """
    Create a reusable TensorFlow dataset.

    Training:
        shuffle -> preprocess -> batch -> prefetch

    Validation/Test:
        preprocess -> batch -> prefetch
    """

    images = np.asarray(images)
    labels = np.asarray(
        labels
    ).reshape(-1).astype(np.int32)

    assert len(images) == len(labels), (
        "Images and labels must contain the same "
        "number of samples."
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (images, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(labels),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        preprocess_sample,
        num_parallel_calls=AUTOTUNE,
        deterministic=not training
    )

    dataset = dataset.batch(
        batch_size,
        drop_remainder=False
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset

Create model-ready datasets

In [ ]:
# =========================================================
# 11. Create final model-ready datasets
# =========================================================

train_dataset = prepare_dataset(
    images=X_train_prepared,
    labels=y_train_prepared,
    batch_size=BATCH_SIZE,
    training=True
)

val_dataset = prepare_dataset(
    images=X_val,
    labels=y_val,
    batch_size=BATCH_SIZE,
    training=False
)

test_dataset = prepare_dataset(
    images=X_test,
    labels=y_test,
    batch_size=BATCH_SIZE,
    training=False
)


print("TensorFlow datasets created successfully.")

In [ ]:
# =========================================================
# 12. Verify the final datasets
# =========================================================

def inspect_dataset(dataset, dataset_name):

    for images, labels in dataset.take(1):

        print(f"\n{dataset_name}")
        print("-" * 60)

        print(
            "Images shape:",
            images.shape
        )

        print(
            "Labels shape:",
            labels.shape
        )

        print(
            "Images dtype:",
            images.dtype
        )

        print(
            "Labels dtype:",
            labels.dtype
        )

        print(
            "Pixel range:",
            f"{tf.reduce_min(images).numpy():.2f}",
            "to",
            f"{tf.reduce_max(images).numpy():.2f}"
        )

        print(
            "Sample labels:",
            labels[:10].numpy()
        )


inspect_dataset(
    train_dataset,
    "Training dataset"
)

inspect_dataset(
    val_dataset,
    "Validation dataset"
)

inspect_dataset(
    test_dataset,
    "Test dataset"
)

# Prepare Image for XAI

In [ ]:
# =========================================================
# XAI SAMPLE IMAGE PREPARATION
# =========================================================
#
# Select one random test image from every class, save each
# selected image as a PNG file, and store its path in:
#
# class_images = {
#     "Basophil": "/.../basophil.png",
#     ...
# }
#
# Dependencies:
#     test_dataset
#     CLASS_NAMES or class_names
#     NUM_CLASSES
# =========================================================

from pathlib import Path
import re

import numpy as np
import tensorflow as tf


# =========================================================
# 1. Configuration
# =========================================================

XAI_RANDOM_SEED = 42

XAI_IMAGE_DIRECTORY = Path(
    "xai_test_images"
)

XAI_IMAGE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# 2. Verify required variables
# =========================================================

xai_required_variables = [
    "test_dataset",
    "NUM_CLASSES"
]

xai_missing_variables = [
    variable_name
    for variable_name in xai_required_variables
    if variable_name not in globals()
]

if xai_missing_variables:
    raise RuntimeError(
        "Run the Dataset Load and Dataset Processing "
        "sections first. "
        f"Missing variables: {xai_missing_variables}"
    )


# Create a reliable local class-name list
if "class_names" in globals():

    xai_class_names = list(class_names)

elif "CLASS_NAMES" in globals():

    if isinstance(CLASS_NAMES, dict):
        xai_class_names = [
            CLASS_NAMES[class_id]
            for class_id in sorted(CLASS_NAMES)
        ]
    else:
        xai_class_names = list(CLASS_NAMES)

else:
    raise RuntimeError(
        "Neither class_names nor CLASS_NAMES is available."
    )


if len(xai_class_names) != NUM_CLASSES:
    raise ValueError(
        f"Expected {NUM_CLASSES} class names, but received "
        f"{len(xai_class_names)}."
    )

In [ ]:
# =========================================================
# 3. Randomly select one test image from every class
# =========================================================

xai_rng = np.random.default_rng(
    XAI_RANDOM_SEED
)

# Stores the selected image tensor for each class ID
selected_images_by_class = {}

# Number of valid images encountered for each class
images_seen_by_class = np.zeros(
    NUM_CLASSES,
    dtype=np.int64
)


# unbatch() makes the code work whether test_dataset was
# originally batched or not.
for image, label in test_dataset.unbatch():

    # -----------------------------------------------------
    # Handle sparse integer labels:
    #     scalar label, for example 3
    #
    # Also supports one-hot labels:
    #     vector label, for example [0, 0, 0, 1, ...]
    # -----------------------------------------------------

    label_array = np.asarray(
        label.numpy()
    )

    if label_array.ndim == 0:
        class_id = int(label_array)
    else:
        class_id = int(
            np.argmax(label_array)
        )


    # Ignore invalid labels defensively
    if not 0 <= class_id < NUM_CLASSES:
        continue


    images_seen_by_class[class_id] += 1

    number_seen = images_seen_by_class[
        class_id
    ]


    # Reservoir sampling:
    # replace the current selection with probability 1/n
    should_select = (
        number_seen == 1
        or xai_rng.integers(number_seen) == 0
    )

    if should_select:
        selected_images_by_class[class_id] = (
            tf.identity(image)
        )

In [ ]:
# =========================================================
# 4. Validate and save the selected images
# =========================================================

missing_class_ids = [
    class_id
    for class_id in range(NUM_CLASSES)
    if class_id not in selected_images_by_class
]

if missing_class_ids:

    missing_class_names = [
        xai_class_names[class_id]
        for class_id in missing_class_ids
    ]

    raise ValueError(
        "The test dataset does not contain a valid image "
        "for every class. Missing classes: "
        f"{missing_class_names}"
    )


def make_safe_filename(class_name):
    """
    Convert a class name into a safe lowercase filename.
    """

    safe_name = class_name.strip().lower()

    safe_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        safe_name
    )

    return safe_name.strip("_")


def prepare_image_for_png(image):
    """
    Convert a test-dataset image into uint8 RGB format
    suitable for PNG encoding.

    Expected output:
        shape: (height, width, 3)
        dtype: uint8
    """

    image = tf.convert_to_tensor(image)

    # Convert to float32 before range handling
    image = tf.cast(
        image,
        tf.float32
    )

    # Add a channel dimension for grayscale images
    if image.shape.rank == 2:
        image = tf.expand_dims(
            image,
            axis=-1
        )

    # Convert grayscale to RGB
    if image.shape[-1] == 1:
        image = tf.image.grayscale_to_rgb(
            image
        )

    # Remove alpha or additional channels if present
    if (
        image.shape[-1] is not None
        and image.shape[-1] > 3
    ):
        image = image[..., :3]


    image_min = float(
        tf.reduce_min(image).numpy()
    )

    image_max = float(
        tf.reduce_max(image).numpy()
    )


    # Handle datasets normalized to [0, 1]
    if image_min >= 0.0 and image_max <= 1.0:
        image = image * 255.0

    # The processing pipeline currently keeps images
    # in the [0, 255] range, so this preserves them.
    image = tf.clip_by_value(
        image,
        0.0,
        255.0
    )

    image = tf.cast(
        tf.round(image),
        tf.uint8
    )

    return image

In [ ]:
# =========================================================
# 5. Save one image per class and build the dictionary
# =========================================================

class_images = {}

for class_id in range(NUM_CLASSES):

    class_name = xai_class_names[class_id]

    selected_image = selected_images_by_class[
        class_id
    ]

    png_image = prepare_image_for_png(
        selected_image
    )

    safe_class_name = make_safe_filename(
        class_name
    )

    image_path = (
        XAI_IMAGE_DIRECTORY
        / f"class_{class_id}_{safe_class_name}.png"
    )

    encoded_image = tf.io.encode_png(
        png_image
    )

    tf.io.write_file(
        str(image_path),
        encoded_image
    )

    # Resolve creates a complete, valid filesystem path
    class_images[class_name] = str(
        image_path.resolve()
    )

In [ ]:
# =========================================================
# 6. Verify XAI image paths
# =========================================================

print("Random test images selected for XAI")
print("-" * 75)

for class_name, image_path in class_images.items():

    path_exists = Path(
        image_path
    ).is_file()

    print(
        f"{class_name:<25} "
        f"Exists: {str(path_exists):<5} "
        f"Path: {image_path}"
    )


assert len(class_images) == NUM_CLASSES, (
    "class_images does not contain every class."
)

assert all(
    Path(image_path).is_file()
    for image_path in class_images.values()
), (
    "At least one XAI image path is invalid."
)


print(
    f"\nSuccessfully stored {len(class_images)} "
    "test images for XAI."
)

# Define Plot Configures

In [ ]:
!pip -q install Lime

In [ ]:
cols             = 4   # fixed 2 columns
rows             = 2   # fixed 2 rows (2x2 grid)

In [ ]:
# =========================================================
# Plot the selected XAI images
# =========================================================
#
# Required variables:
#     class_images
#     cols
#     rows
# =========================================================

import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path


# Verify required variables
required_plot_variables = [
    "class_images",
    "cols",
    "rows"
]

missing_plot_variables = [
    variable_name
    for variable_name in required_plot_variables
    if variable_name not in globals()
]

if missing_plot_variables:
    raise RuntimeError(
        "Missing required variables: "
        f"{missing_plot_variables}"
    )


# Ensure the grid has enough positions
number_of_images = len(class_images)

if rows * cols < number_of_images:
    raise ValueError(
        f"The {rows} × {cols} grid contains only "
        f"{rows * cols} positions, but class_images "
        f"contains {number_of_images} images."
    )


# Create the figure
figure, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows)
)

# Convert axes into a one-dimensional array.
# This also works when rows = 1 or cols = 1.
axes = np.asarray(axes).reshape(-1)


# Plot every selected class image
for plot_index, (class_name, image_path) in enumerate(
    class_images.items()
):

    image_path = Path(image_path)

    if not image_path.is_file():
        raise FileNotFoundError(
            f"Image file not found for {class_name}: "
            f"{image_path}"
        )

    with Image.open(image_path) as loaded_image:
        image = loaded_image.convert("RGB")

        axes[plot_index].imshow(image)

    axes[plot_index].set_title(
        class_name,
        fontsize=12,
        fontweight="bold"
    )

    axes[plot_index].axis("off")


# Hide any unused grid positions
for unused_index in range(
    number_of_images,
    len(axes)
):
    axes[unused_index].axis("off")


figure.suptitle(
    "Random Test Images Selected for XAI",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
plt.show()

# Model 1: DenseNet121

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
# Load the DenseNet121 model without the top layer:
base_model = DenseNet121(input_shape=(224,224,3), include_top=False, weights='imagenet')

model_name = 'DenseNet121'

In [ ]:
# Freeze the pre-trained layers
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
# Create a Sequential model
x = base_model.output
x = Dropout(0.3)(x)  # Add a dropout layer
x = GlobalAveragePooling2D()(x)

predictions = Dense(len(class_names), activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# Compile the model:
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
import time
from tensorflow.keras.callbacks import EarlyStopping

# Define early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

# Record training start time
training_start_time = time.perf_counter()

# train the model (replace this with your model)
history = model.fit(train_dataset, validation_data=val_dataset, epochs=30, callbacks=[early_stopping])

# Record training end time
training_end_time = time.perf_counter()

# Calculate total training time in minutes
total_training_minutes = (training_end_time - training_start_time) / 60


print(f"\nTotal training time: {total_training_minutes:.2f} minutes")


 Evaluation

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef
)

# 1. Evaluate model
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=1)

# 2. True labels
test_true_labels = np.concatenate([labels.numpy().ravel() for _, labels in test_dataset]).astype(np.int32)

# 3. Predictions
test_pred_probs = model.predict(test_dataset, verbose=1)
test_pred_labels = np.argmax(test_pred_probs, axis=1)

# 4. Metrics
precision = precision_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
recall = recall_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
f1 = f1_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
cm = confusion_matrix(test_true_labels, test_pred_labels, labels=np.arange(NUM_CLASSES))

# Specificity
specificity_scores = []
for i in range(NUM_CLASSES):
    tp, fn = cm[i, i], cm[i, :].sum() - cm[i, i]
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    specificity_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
avg_specificity = np.mean(specificity_scores)

# Cohen’s Kappa & MCC
kappa = cohen_kappa_score(test_true_labels, test_pred_labels, labels=np.arange(NUM_CLASSES))
mcc = matthews_corrcoef(test_true_labels, test_pred_labels)

# Results
print(f"\n{model_name} Performance Metrics\n" + "-"*55)
print(f"Test Loss: {test_loss:.2f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Macro Precision: {precision*100:.2f}%")
print(f"Macro Recall: {recall*100:.2f}%")
print(f"Macro F1-Score: {f1*100:.2f}%")
print(f"Average Specificity: {avg_specificity*100:.2f}%")
print(f"Cohen's Kappa: {kappa:.2f}")
print(f"Matthews Corrcoef (MCC): {mcc:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], marker='o', label='Training Loss')
plt.plot(history.history['val_loss'], marker='o', label='Validation Loss')

plt.title(f'{model_name} - Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Add grid with black color
plt.grid(True, color='black', linestyle='--', alpha=0.7)

# Ensure borders (spines) are visible and styled
for spine in plt.gca().spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)
    spine.set_color("black")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# Generate confusion matrix
conf_matrix = confusion_matrix(test_true_labels, test_pred_labels)

# Global percentages (counts relative to total samples)
conf_percent = conf_matrix.astype('float') / conf_matrix.sum() * 100

# Annotate each cell with "count\nxx.xx%"
annot_text = np.array([
    [f"{conf_matrix[i, j]}\n{conf_percent[i, j]:.2f}%" for j in range(conf_matrix.shape[1])]
    for i in range(conf_matrix.shape[0])
])

# Plot confusion matrix
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    conf_matrix,
    annot=annot_text,
    fmt='',
    cmap='Blues',
    cbar=True,
    xticklabels=class_names,
    yticklabels=class_names,
    annot_kws={"size": 14, "weight": "bold"},
    linecolor='black',
    linewidths=1
)

# Adjust text color dynamically
for text in ax.texts:
    x, y = text.get_position()
    i, j = int(y - 0.5), int(x - 0.5)
    facecolor = ax.collections[0].get_facecolor()[i * conf_matrix.shape[1] + j]
    r, g, b, _ = facecolor
    brightness = (r + g + b) / 3
    text.set_color("white" if brightness < 0.5 else "black")

# Labels and title
plt.xlabel('Predicted labels', fontsize=12, fontweight="bold")
plt.ylabel('True labels', fontsize=12, fontweight="bold")
plt.title(f'{model_name} - Confusion Matrix', fontsize=16, fontweight="bold")

plt.xticks(rotation=45, ha="right", fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Interactive mode (works in Jupyter/Notebook; in scripts use plt.show(block=False))
plt.ion()

y_true = np.asarray(test_true_labels)
y_score = np.asarray(test_pred_probs)
class_names = list(class_names)
n_classes = len(class_names)
model_name = str(model_name)

y_true_bin = label_binarize(y_true, classes=range(n_classes))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.close("all")
fig, ax = plt.subplots(figsize=(6, 5))

# Light blue background
ax.set_facecolor("aliceblue")

cmap = plt.get_cmap("tab10" if n_classes <= 10 else "tab20")
colors = [cmap(i % cmap.N) for i in range(n_classes)]

# Diagonal reference
ax.plot([0, 1], [0, 1], "--", linewidth=1.3, color="0.4", zorder=1)

# ROC curves
for i, cname in enumerate(class_names):
    ax.plot(fpr[i], tpr[i],
            linewidth=2.2,
            color=colors[i],
            label=f"{cname} (AUC={roc_auc[i]:.3f})",
            zorder=3)

# Axis limits
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Labels
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title(f"{model_name} — ROC Curves",
             fontsize=12, fontweight="bold", pad=10)

# ---- Structured Grid ----
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_xticks(np.arange(0, 1.01, 0.05), minor=True)
ax.set_yticks(np.arange(0, 1.01, 0.05), minor=True)

ax.grid(which="major", linestyle="-", linewidth=0.8, alpha=0.4)
ax.grid(which="minor", linestyle=":", linewidth=0.6, alpha=0.35)
ax.set_axisbelow(True)

# Spines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(1.0)

# Legend outside, bottom center
legend = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.25),   # bottom center outside
    frameon=False,
    fontsize=8.5,
    ncol=2
)

plt.tight_layout()
plt.show()


Model Info

In [ ]:
model.summary()

XAI

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D

# Find the last convolutional layer in the full model
for layer in reversed(model.layers):  # Start from the end
    if isinstance(layer, Conv2D):  # Check if the layer is a convolutional layer
        print(f"Last Conv Layer: {layer.name}, Output Shape: {layer.output.shape}")
        break

# --- Use Grad-CAM for each class image ---
last_conv_layer_name = layer.name  # replace with printed layer name

Grad-CAM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image


def gradcam(img_path, model, last_conv_layer_name, size=(224, 224)):
    # Load image
    img = image.load_img(img_path, target_size=size, color_mode="rgb")
    img_array = np.expand_dims(image.img_to_array(img), axis=0)

    # Model returning convolution output and predictions
    grad_model = Model(
        model.inputs,
        [
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    # Calculate gradients
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array, training=False)
        predicted_class = tf.argmax(predictions[0])
        class_score = predictions[:, predicted_class]

    gradients = tape.gradient(class_score, conv_output)
    weights = tf.reduce_mean(gradients, axis=(0, 1, 2))

    # Create heatmap
    heatmap = tf.reduce_sum(conv_output[0] * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + tf.keras.backend.epsilon()

    # Original image in [0, 1]
    original = image.img_to_array(
        image.load_img(img_path, color_mode="rgb")
    ) / 255.0

    # Resize and color heatmap
    heatmap = tf.image.resize(
        heatmap[..., None],
        original.shape[:2]
    ).numpy().squeeze()

    colored_heatmap = mpl.colormaps["jet"](
        np.clip(heatmap, 0, 1)
    )[..., :3]

    # Blend and clip
    overlay = np.clip(
        0.6 * original + 0.4 * colored_heatmap,
        0,
        1
    )

    return (
        overlay,
        int(predicted_class.numpy()),
        float(tf.reduce_max(predictions[0]).numpy())
    )


# Ensure the grid is large enough
if rows * cols < len(class_images):
    raise ValueError("rows × cols is smaller than the number of class images.")


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows)
)

axes = np.asarray(axes).reshape(-1)


for i, (true_class, img_path) in enumerate(class_images.items()):
    overlay, pred_id, confidence = gradcam(
        img_path,
        model,
        last_conv_layer_name,
        size=(IMG_HEIGHT, IMG_WIDTH)
    )

    axes[i].imshow(overlay)
    axes[i].set_title(
        f"True: {true_class}\n"
        f"Pred: {class_names[pred_id]} ({confidence:.1%})",
        fontsize=10
    )
    axes[i].axis("off")


# Hide unused axes
for i in range(len(class_images), len(axes)):
    axes[i].axis("off")


plt.suptitle(
    f"{model_name} Grad-CAM",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

Lime

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from lime import lime_image
from skimage.segmentation import mark_boundaries
from skimage.segmentation import slic


# =========================================================
# Settings
# =========================================================

input_size = (224, 224)
num_samples = 1000
batch_size = 64
num_features_vis = 8
positive_only = True
hide_color = 0


# Enable dynamic GPU memory allocation
gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("GPU available:", len(gpus) > 0)


# =========================================================
# Helpers
# =========================================================

def load_rgb(path):
    image_bgr = cv2.imread(path)

    if image_bgr is None:
        raise FileNotFoundError(f"Could not load image: {path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    return cv2.resize(
        image_rgb,
        input_size,
        interpolation=cv2.INTER_AREA
    )


def predict_fn(images):
    """
    LIME passes RGB images with values near 0–255.

    The model performs its own preprocessing internally.
    TensorFlow inference will use the GPU when available.
    """
    images = tf.convert_to_tensor(
        images,
        dtype=tf.float32
    )

    predictions = model(
        images,
        training=False
    )

    return predictions.numpy()


def create_lime_overlay(explainer, rgb_image):
    prediction = predict_fn(
        rgb_image[np.newaxis, ...]
    )[0]

    predicted_index = int(
        np.argmax(prediction)
    )

    explanation = explainer.explain_instance(
        image=rgb_image.astype(np.double),
        classifier_fn=predict_fn,
        top_labels=1,
        labels=(predicted_index,),
        hide_color=hide_color,
        num_samples=num_samples,
        batch_size=batch_size,
        segmentation_fn=lambda image: slic(
            image,
            n_segments=50,
            compactness=10,
            sigma=1,
            start_label=0
        ),
        random_seed=42
    )

    explained_image, mask = explanation.get_image_and_mask(
        label=predicted_index,
        positive_only=positive_only,
        num_features=num_features_vis,
        hide_rest=False
    )

    # mark_boundaries expects a float image in [0, 1]
    explained_image = np.clip(
        explained_image / 255.0,
        0.0,
        1.0
    )

    overlay = mark_boundaries(
        explained_image,
        mask,
        color=(1, 1, 0),
        mode="thick"
    )

    return (
        np.clip(overlay, 0.0, 1.0),
        predicted_index,
        float(prediction[predicted_index])
    )


# =========================================================
# Generate LIME explanations
# =========================================================

explainer = lime_image.LimeImageExplainer(
    random_state=42
)

lime_results = []

for true_class, path in class_images.items():
    rgb_image = load_rgb(path)

    overlay, predicted_index, confidence = (
        create_lime_overlay(
            explainer,
            rgb_image
        )
    )

    predicted_name = class_names[
        predicted_index
    ]

    lime_results.append({
        "image": overlay,
        "true_class": true_class,
        "predicted_class": predicted_name,
        "confidence": confidence
    })


# =========================================================
# Plot explanations
# =========================================================

if rows * cols < len(lime_results):
    raise ValueError(
        f"The {rows} × {cols} grid has only "
        f"{rows * cols} positions for "
        f"{len(lime_results)} images."
    )


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows),
    dpi=150
)

axes = np.asarray(axes).reshape(-1)


for index, result in enumerate(lime_results):
    axes[index].imshow(
        result["image"]
    )

    axes[index].set_title(
        f"True: {result['true_class']}\n"
        f"Pred: {result['predicted_class']} "
        f"({result['confidence']:.1%})",
        fontsize=9,
        fontweight="bold"
    )

    axes[index].axis("off")


for index in range(
    len(lime_results),
    len(axes)
):
    axes[index].axis("off")


fig.suptitle(
    f"{model_name} LIME Explanations",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# Model 2: ResNet50

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
# Load the ResNet50 model without the top layer:
base_model = ResNet50(input_shape=(224,224,3), include_top=False, weights='imagenet')

model_name = 'ResNet50'

In [ ]:
# Freeze the pre-trained layers
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
# Create a Sequential model
x = base_model.output
x = Dropout(0.3)(x)  # Add a dropout layer
x = GlobalAveragePooling2D()(x)

predictions = Dense(len(class_names), activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# Compile the model:
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
import time
from tensorflow.keras.callbacks import EarlyStopping

# Define early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

# Record training start time
training_start_time = time.perf_counter()

# train the model (replace this with your model)
history = model.fit(train_dataset, validation_data=val_dataset, epochs=30, callbacks=[early_stopping])

# Record training end time
training_end_time = time.perf_counter()

# Calculate total training time in minutes
total_training_minutes = (training_end_time - training_start_time) / 60


print(f"\nTotal training time: {total_training_minutes:.2f} minutes")


# Evaluation

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef
)

# 1. Evaluate model
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=1)

# 2. True labels
test_true_labels = np.concatenate([labels.numpy().ravel() for _, labels in test_dataset]).astype(np.int32)

# 3. Predictions
test_pred_probs = model.predict(test_dataset, verbose=1)
test_pred_labels = np.argmax(test_pred_probs, axis=1)

# 4. Metrics
precision = precision_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
recall = recall_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
f1 = f1_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
cm = confusion_matrix(test_true_labels, test_pred_labels, labels=np.arange(NUM_CLASSES))

# Specificity
specificity_scores = []
for i in range(NUM_CLASSES):
    tp, fn = cm[i, i], cm[i, :].sum() - cm[i, i]
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    specificity_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
avg_specificity = np.mean(specificity_scores)

# Cohen’s Kappa & MCC
kappa = cohen_kappa_score(test_true_labels, test_pred_labels, labels=np.arange(NUM_CLASSES))
mcc = matthews_corrcoef(test_true_labels, test_pred_labels)

# Results
print(f"\n{model_name} Performance Metrics\n" + "-"*55)
print(f"Test Loss: {test_loss:.2f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Macro Precision: {precision*100:.2f}%")
print(f"Macro Recall: {recall*100:.2f}%")
print(f"Macro F1-Score: {f1*100:.2f}%")
print(f"Average Specificity: {avg_specificity*100:.2f}%")
print(f"Cohen's Kappa: {kappa:.2f}")
print(f"Matthews Corrcoef (MCC): {mcc:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], marker='o', label='Training Loss')
plt.plot(history.history['val_loss'], marker='o', label='Validation Loss')

plt.title(f'{model_name} - Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Add grid with black color
plt.grid(True, color='black', linestyle='--', alpha=0.7)

# Ensure borders (spines) are visible and styled
for spine in plt.gca().spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)
    spine.set_color("black")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# Generate confusion matrix
conf_matrix = confusion_matrix(test_true_labels, test_pred_labels)

# Global percentages (counts relative to total samples)
conf_percent = conf_matrix.astype('float') / conf_matrix.sum() * 100

# Annotate each cell with "count\nxx.xx%"
annot_text = np.array([
    [f"{conf_matrix[i, j]}\n{conf_percent[i, j]:.2f}%" for j in range(conf_matrix.shape[1])]
    for i in range(conf_matrix.shape[0])
])

# Plot confusion matrix
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    conf_matrix,
    annot=annot_text,
    fmt='',
    cmap='Blues',
    cbar=True,
    xticklabels=class_names,
    yticklabels=class_names,
    annot_kws={"size": 14, "weight": "bold"},
    linecolor='black',
    linewidths=1
)

# Adjust text color dynamically
for text in ax.texts:
    x, y = text.get_position()
    i, j = int(y - 0.5), int(x - 0.5)
    facecolor = ax.collections[0].get_facecolor()[i * conf_matrix.shape[1] + j]
    r, g, b, _ = facecolor
    brightness = (r + g + b) / 3
    text.set_color("white" if brightness < 0.5 else "black")

# Labels and title
plt.xlabel('Predicted labels', fontsize=12, fontweight="bold")
plt.ylabel('True labels', fontsize=12, fontweight="bold")
plt.title(f'{model_name} - Confusion Matrix', fontsize=16, fontweight="bold")

plt.xticks(rotation=45, ha="right", fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Interactive mode (works in Jupyter/Notebook; in scripts use plt.show(block=False))
plt.ion()

y_true = np.asarray(test_true_labels)
y_score = np.asarray(test_pred_probs)
class_names = list(class_names)
n_classes = len(class_names)
model_name = str(model_name)

y_true_bin = label_binarize(y_true, classes=range(n_classes))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.close("all")
fig, ax = plt.subplots(figsize=(6, 5))

# Light blue background
ax.set_facecolor("aliceblue")

cmap = plt.get_cmap("tab10" if n_classes <= 10 else "tab20")
colors = [cmap(i % cmap.N) for i in range(n_classes)]

# Diagonal reference
ax.plot([0, 1], [0, 1], "--", linewidth=1.3, color="0.4", zorder=1)

# ROC curves
for i, cname in enumerate(class_names):
    ax.plot(fpr[i], tpr[i],
            linewidth=2.2,
            color=colors[i],
            label=f"{cname} (AUC={roc_auc[i]:.3f})",
            zorder=3)

# Axis limits
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Labels
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title(f"{model_name} — ROC Curves",
             fontsize=12, fontweight="bold", pad=10)

# ---- Structured Grid ----
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_xticks(np.arange(0, 1.01, 0.05), minor=True)
ax.set_yticks(np.arange(0, 1.01, 0.05), minor=True)

ax.grid(which="major", linestyle="-", linewidth=0.8, alpha=0.4)
ax.grid(which="minor", linestyle=":", linewidth=0.6, alpha=0.35)
ax.set_axisbelow(True)

# Spines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(1.0)

# Legend outside, bottom center
legend = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.25),   # bottom center outside
    frameon=False,
    fontsize=8.5,
    ncol=2
)

plt.tight_layout()
plt.show()


# Model Info

In [ ]:
model.summary()

# XAI

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D

# Find the last convolutional layer in the full model
for layer in reversed(model.layers):  # Start from the end
    if isinstance(layer, Conv2D):  # Check if the layer is a convolutional layer
        print(f"Last Conv Layer: {layer.name}, Output Shape: {layer.output.shape}")
        break

# --- Use Grad-CAM for each class image ---
last_conv_layer_name = layer.name  # replace with printed layer name

# Grad-CAM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image


def gradcam(img_path, model, last_conv_layer_name, size=(224, 224)):
    # Load image
    img = image.load_img(img_path, target_size=size, color_mode="rgb")
    img_array = np.expand_dims(image.img_to_array(img), axis=0)

    # Model returning convolution output and predictions
    grad_model = Model(
        model.inputs,
        [
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    # Calculate gradients
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array, training=False)
        predicted_class = tf.argmax(predictions[0])
        class_score = predictions[:, predicted_class]

    gradients = tape.gradient(class_score, conv_output)
    weights = tf.reduce_mean(gradients, axis=(0, 1, 2))

    # Create heatmap
    heatmap = tf.reduce_sum(conv_output[0] * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + tf.keras.backend.epsilon()

    # Original image in [0, 1]
    original = image.img_to_array(
        image.load_img(img_path, color_mode="rgb")
    ) / 255.0

    # Resize and color heatmap
    heatmap = tf.image.resize(
        heatmap[..., None],
        original.shape[:2]
    ).numpy().squeeze()

    colored_heatmap = mpl.colormaps["jet"](
        np.clip(heatmap, 0, 1)
    )[..., :3]

    # Blend and clip
    overlay = np.clip(
        0.6 * original + 0.4 * colored_heatmap,
        0,
        1
    )

    return (
        overlay,
        int(predicted_class.numpy()),
        float(tf.reduce_max(predictions[0]).numpy())
    )


# Ensure the grid is large enough
if rows * cols < len(class_images):
    raise ValueError("rows × cols is smaller than the number of class images.")


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows)
)

axes = np.asarray(axes).reshape(-1)


for i, (true_class, img_path) in enumerate(class_images.items()):
    overlay, pred_id, confidence = gradcam(
        img_path,
        model,
        last_conv_layer_name,
        size=(IMG_HEIGHT, IMG_WIDTH)
    )

    axes[i].imshow(overlay)
    axes[i].set_title(
        f"True: {true_class}\n"
        f"Pred: {class_names[pred_id]} ({confidence:.1%})",
        fontsize=10
    )
    axes[i].axis("off")


# Hide unused axes
for i in range(len(class_images), len(axes)):
    axes[i].axis("off")


plt.suptitle(
    f"{model_name} Grad-CAM",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# Lime

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from lime import lime_image
from skimage.segmentation import mark_boundaries
from skimage.segmentation import slic


# =========================================================
# Settings
# =========================================================

input_size = (224, 224)
num_samples = 1000
batch_size = 64
num_features_vis = 8
positive_only = True
hide_color = 0


# Enable dynamic GPU memory allocation
gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("GPU available:", len(gpus) > 0)


# =========================================================
# Helpers
# =========================================================

def load_rgb(path):
    image_bgr = cv2.imread(path)

    if image_bgr is None:
        raise FileNotFoundError(f"Could not load image: {path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    return cv2.resize(
        image_rgb,
        input_size,
        interpolation=cv2.INTER_AREA
    )


def predict_fn(images):
    """
    LIME passes RGB images with values near 0–255.

    The model performs its own preprocessing internally.
    TensorFlow inference will use the GPU when available.
    """
    images = tf.convert_to_tensor(
        images,
        dtype=tf.float32
    )

    predictions = model(
        images,
        training=False
    )

    return predictions.numpy()


def create_lime_overlay(explainer, rgb_image):
    prediction = predict_fn(
        rgb_image[np.newaxis, ...]
    )[0]

    predicted_index = int(
        np.argmax(prediction)
    )

    explanation = explainer.explain_instance(
        image=rgb_image.astype(np.double),
        classifier_fn=predict_fn,
        top_labels=1,
        labels=(predicted_index,),
        hide_color=hide_color,
        num_samples=num_samples,
        batch_size=batch_size,
        segmentation_fn=lambda image: slic(
            image,
            n_segments=50,
            compactness=10,
            sigma=1,
            start_label=0
        ),
        random_seed=42
    )

    explained_image, mask = explanation.get_image_and_mask(
        label=predicted_index,
        positive_only=positive_only,
        num_features=num_features_vis,
        hide_rest=False
    )

    # mark_boundaries expects a float image in [0, 1]
    explained_image = np.clip(
        explained_image / 255.0,
        0.0,
        1.0
    )

    overlay = mark_boundaries(
        explained_image,
        mask,
        color=(1, 1, 0),
        mode="thick"
    )

    return (
        np.clip(overlay, 0.0, 1.0),
        predicted_index,
        float(prediction[predicted_index])
    )


# =========================================================
# Generate LIME explanations
# =========================================================

explainer = lime_image.LimeImageExplainer(
    random_state=42
)

lime_results = []

for true_class, path in class_images.items():
    rgb_image = load_rgb(path)

    overlay, predicted_index, confidence = (
        create_lime_overlay(
            explainer,
            rgb_image
        )
    )

    predicted_name = class_names[
        predicted_index
    ]

    lime_results.append({
        "image": overlay,
        "true_class": true_class,
        "predicted_class": predicted_name,
        "confidence": confidence
    })


# =========================================================
# Plot explanations
# =========================================================

if rows * cols < len(lime_results):
    raise ValueError(
        f"The {rows} × {cols} grid has only "
        f"{rows * cols} positions for "
        f"{len(lime_results)} images."
    )


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows),
    dpi=150
)

axes = np.asarray(axes).reshape(-1)


for index, result in enumerate(lime_results):
    axes[index].imshow(
        result["image"]
    )

    axes[index].set_title(
        f"True: {result['true_class']}\n"
        f"Pred: {result['predicted_class']} "
        f"({result['confidence']:.1%})",
        fontsize=9,
        fontweight="bold"
    )

    axes[index].axis("off")


for index in range(
    len(lime_results),
    len(axes)
):
    axes[index].axis("off")


fig.suptitle(
    f"{model_name} LIME Explanations",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# Model 3: EfficientNetB0

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
# Load the EfficientNetB0 model without the top layer:
base_model = EfficientNetB0(input_shape=(224,224,3), include_top=False, weights='imagenet')

model_name = 'EfficientNetB0'

In [ ]:
# Freeze the pre-trained layers
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
# Create a Sequential model
x = base_model.output
x = Dropout(0.3)(x)  # Add a dropout layer
x = GlobalAveragePooling2D()(x)

predictions = Dense(len(class_names), activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# Compile the model:
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
import time
from tensorflow.keras.callbacks import EarlyStopping

# Define early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

# Record training start time
training_start_time = time.perf_counter()

# train the model (replace this with your model)
history = model.fit(train_dataset, validation_data=val_dataset, epochs=30, callbacks=[early_stopping])

# Record training end time
training_end_time = time.perf_counter()

# Calculate total training time in minutes
total_training_minutes = (training_end_time - training_start_time) / 60


print(f"\nTotal training time: {total_training_minutes:.2f} minutes")


# Evaluation

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef
)

# 1. Evaluate model
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=1)

# 2. True labels
test_true_labels = np.concatenate([labels.numpy().ravel() for _, labels in test_dataset]).astype(np.int32)

# 3. Predictions
test_pred_probs = model.predict(test_dataset, verbose=1)
test_pred_labels = np.argmax(test_pred_probs, axis=1)

# 4. Metrics
precision = precision_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
recall = recall_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
f1 = f1_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
cm = confusion_matrix(test_true_labels, test_pred_labels, labels=np.arange(NUM_CLASSES))

# Specificity
specificity_scores = []
for i in range(NUM_CLASSES):
    tp, fn = cm[i, i], cm[i, :].sum() - cm[i, i]
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    specificity_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
avg_specificity = np.mean(specificity_scores)

# Cohen’s Kappa & MCC
kappa = cohen_kappa_score(test_true_labels, test_pred_labels, labels=np.arange(NUM_CLASSES))
mcc = matthews_corrcoef(test_true_labels, test_pred_labels)

# Results
print(f"\n{model_name} Performance Metrics\n" + "-"*55)
print(f"Test Loss: {test_loss:.2f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Macro Precision: {precision*100:.2f}%")
print(f"Macro Recall: {recall*100:.2f}%")
print(f"Macro F1-Score: {f1*100:.2f}%")
print(f"Average Specificity: {avg_specificity*100:.2f}%")
print(f"Cohen's Kappa: {kappa:.2f}")
print(f"Matthews Corrcoef (MCC): {mcc:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], marker='o', label='Training Loss')
plt.plot(history.history['val_loss'], marker='o', label='Validation Loss')

plt.title(f'{model_name} - Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Add grid with black color
plt.grid(True, color='black', linestyle='--', alpha=0.7)

# Ensure borders (spines) are visible and styled
for spine in plt.gca().spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)
    spine.set_color("black")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# Generate confusion matrix
conf_matrix = confusion_matrix(test_true_labels, test_pred_labels)

# Global percentages (counts relative to total samples)
conf_percent = conf_matrix.astype('float') / conf_matrix.sum() * 100

# Annotate each cell with "count\nxx.xx%"
annot_text = np.array([
    [f"{conf_matrix[i, j]}\n{conf_percent[i, j]:.2f}%" for j in range(conf_matrix.shape[1])]
    for i in range(conf_matrix.shape[0])
])

# Plot confusion matrix
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    conf_matrix,
    annot=annot_text,
    fmt='',
    cmap='Blues',
    cbar=True,
    xticklabels=class_names,
    yticklabels=class_names,
    annot_kws={"size": 14, "weight": "bold"},
    linecolor='black',
    linewidths=1
)

# Adjust text color dynamically
for text in ax.texts:
    x, y = text.get_position()
    i, j = int(y - 0.5), int(x - 0.5)
    facecolor = ax.collections[0].get_facecolor()[i * conf_matrix.shape[1] + j]
    r, g, b, _ = facecolor
    brightness = (r + g + b) / 3
    text.set_color("white" if brightness < 0.5 else "black")

# Labels and title
plt.xlabel('Predicted labels', fontsize=12, fontweight="bold")
plt.ylabel('True labels', fontsize=12, fontweight="bold")
plt.title(f'{model_name} - Confusion Matrix', fontsize=16, fontweight="bold")

plt.xticks(rotation=45, ha="right", fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Interactive mode (works in Jupyter/Notebook; in scripts use plt.show(block=False))
plt.ion()

y_true = np.asarray(test_true_labels)
y_score = np.asarray(test_pred_probs)
class_names = list(class_names)
n_classes = len(class_names)
model_name = str(model_name)

y_true_bin = label_binarize(y_true, classes=range(n_classes))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.close("all")
fig, ax = plt.subplots(figsize=(6, 5))

# Light blue background
ax.set_facecolor("aliceblue")

cmap = plt.get_cmap("tab10" if n_classes <= 10 else "tab20")
colors = [cmap(i % cmap.N) for i in range(n_classes)]

# Diagonal reference
ax.plot([0, 1], [0, 1], "--", linewidth=1.3, color="0.4", zorder=1)

# ROC curves
for i, cname in enumerate(class_names):
    ax.plot(fpr[i], tpr[i],
            linewidth=2.2,
            color=colors[i],
            label=f"{cname} (AUC={roc_auc[i]:.3f})",
            zorder=3)

# Axis limits
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Labels
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title(f"{model_name} — ROC Curves",
             fontsize=12, fontweight="bold", pad=10)

# ---- Structured Grid ----
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_xticks(np.arange(0, 1.01, 0.05), minor=True)
ax.set_yticks(np.arange(0, 1.01, 0.05), minor=True)

ax.grid(which="major", linestyle="-", linewidth=0.8, alpha=0.4)
ax.grid(which="minor", linestyle=":", linewidth=0.6, alpha=0.35)
ax.set_axisbelow(True)

# Spines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(1.0)

# Legend outside, bottom center
legend = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.25),   # bottom center outside
    frameon=False,
    fontsize=8.5,
    ncol=2
)

plt.tight_layout()
plt.show()


# Model Info

In [ ]:
model.summary()

# XAI

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D

# Find the last convolutional layer in the full model
for layer in reversed(model.layers):  # Start from the end
    if isinstance(layer, Conv2D):  # Check if the layer is a convolutional layer
        print(f"Last Conv Layer: {layer.name}, Output Shape: {layer.output.shape}")
        break

# --- Use Grad-CAM for each class image ---
last_conv_layer_name = layer.name  # replace with printed layer name

# Grad-CAM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image


def gradcam(img_path, model, last_conv_layer_name, size=(224, 224)):
    # Load image
    img = image.load_img(img_path, target_size=size, color_mode="rgb")
    img_array = np.expand_dims(image.img_to_array(img), axis=0)

    # Model returning convolution output and predictions
    grad_model = Model(
        model.inputs,
        [
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    # Calculate gradients
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array, training=False)
        predicted_class = tf.argmax(predictions[0])
        class_score = predictions[:, predicted_class]

    gradients = tape.gradient(class_score, conv_output)
    weights = tf.reduce_mean(gradients, axis=(0, 1, 2))

    # Create heatmap
    heatmap = tf.reduce_sum(conv_output[0] * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + tf.keras.backend.epsilon()

    # Original image in [0, 1]
    original = image.img_to_array(
        image.load_img(img_path, color_mode="rgb")
    ) / 255.0

    # Resize and color heatmap
    heatmap = tf.image.resize(
        heatmap[..., None],
        original.shape[:2]
    ).numpy().squeeze()

    colored_heatmap = mpl.colormaps["jet"](
        np.clip(heatmap, 0, 1)
    )[..., :3]

    # Blend and clip
    overlay = np.clip(
        0.6 * original + 0.4 * colored_heatmap,
        0,
        1
    )

    return (
        overlay,
        int(predicted_class.numpy()),
        float(tf.reduce_max(predictions[0]).numpy())
    )


# Ensure the grid is large enough
if rows * cols < len(class_images):
    raise ValueError("rows × cols is smaller than the number of class images.")


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows)
)

axes = np.asarray(axes).reshape(-1)


for i, (true_class, img_path) in enumerate(class_images.items()):
    overlay, pred_id, confidence = gradcam(
        img_path,
        model,
        last_conv_layer_name,
        size=(IMG_HEIGHT, IMG_WIDTH)
    )

    axes[i].imshow(overlay)
    axes[i].set_title(
        f"True: {true_class}\n"
        f"Pred: {class_names[pred_id]} ({confidence:.1%})",
        fontsize=10
    )
    axes[i].axis("off")


# Hide unused axes
for i in range(len(class_images), len(axes)):
    axes[i].axis("off")


plt.suptitle(
    f"{model_name} Grad-CAM",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# Lime

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from lime import lime_image
from skimage.segmentation import mark_boundaries
from skimage.segmentation import slic


# =========================================================
# Settings
# =========================================================

input_size = (224, 224)
num_samples = 1000
batch_size = 64
num_features_vis = 8
positive_only = True
hide_color = 0


# =========================================================
# CPU configuration
# =========================================================

LIME_DEVICE = "/CPU:0"

print("LIME inference device:", LIME_DEVICE)


# =========================================================
# Helpers
# =========================================================

def load_rgb(path):
    image_bgr = cv2.imread(path)

    if image_bgr is None:
        raise FileNotFoundError(
            f"Could not load image: {path}"
        )

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    return cv2.resize(
        image_rgb,
        input_size,
        interpolation=cv2.INTER_AREA
    )


def predict_fn(images):
    """
    LIME passes batches of RGB images with values near 0–255.

    All model inference performed for LIME is explicitly
    placed on the CPU.
    """
    images = np.asarray(
        images,
        dtype=np.float32
    )

    with tf.device(LIME_DEVICE):
        images = tf.convert_to_tensor(
            images,
            dtype=tf.float32
        )

        predictions = model(
            images,
            training=False
        )

    return predictions.numpy()


def create_lime_overlay(explainer, rgb_image):
    prediction = predict_fn(
        rgb_image[np.newaxis, ...]
    )[0]

    predicted_index = int(
        np.argmax(prediction)
    )

    explanation = explainer.explain_instance(
        image=rgb_image.astype(np.double),
        classifier_fn=predict_fn,
        top_labels=1,
        labels=(predicted_index,),
        hide_color=hide_color,
        num_samples=num_samples,
        batch_size=batch_size,
        segmentation_fn=lambda image: slic(
            image,
            n_segments=50,
            compactness=10,
            sigma=1,
            start_label=0
        ),
        random_seed=42
    )

    explained_image, mask = explanation.get_image_and_mask(
        label=predicted_index,
        positive_only=positive_only,
        num_features=num_features_vis,
        hide_rest=False
    )

    # mark_boundaries expects a float image in [0, 1].
    explained_image = np.clip(
        explained_image / 255.0,
        0.0,
        1.0
    )

    overlay = mark_boundaries(
        explained_image,
        mask,
        color=(1, 1, 0),
        mode="thick"
    )

    return (
        np.clip(overlay, 0.0, 1.0),
        predicted_index,
        float(prediction[predicted_index])
    )


# =========================================================
# Generate LIME explanations
# =========================================================

explainer = lime_image.LimeImageExplainer(
    random_state=42
)

lime_results = []

for true_class, path in class_images.items():
    print(f"Generating CPU LIME explanation: {true_class}")

    rgb_image = load_rgb(path)

    overlay, predicted_index, confidence = (
        create_lime_overlay(
            explainer,
            rgb_image
        )
    )

    predicted_name = class_names[
        predicted_index
    ]

    lime_results.append({
        "image": overlay,
        "true_class": true_class,
        "predicted_class": predicted_name,
        "confidence": confidence
    })


# =========================================================
# Plot explanations
# =========================================================

if rows * cols < len(lime_results):
    raise ValueError(
        f"The {rows} × {cols} grid has only "
        f"{rows * cols} positions for "
        f"{len(lime_results)} images."
    )


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows),
    dpi=150
)

axes = np.asarray(axes).reshape(-1)


for index, result in enumerate(lime_results):
    axes[index].imshow(
        result["image"]
    )

    axes[index].set_title(
        f"True: {result['true_class']}\n"
        f"Pred: {result['predicted_class']} "
        f"({result['confidence']:.1%})",
        fontsize=9,
        fontweight="bold"
    )

    axes[index].axis("off")


for index in range(
    len(lime_results),
    len(axes)
):
    axes[index].axis("off")


fig.suptitle(
    f"{model_name} LIME Explanations",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# Model 4: MobileNetV3Large

In [ ]:
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [ ]:
# Load the MobileNetV3Large model without the top layer:
base_model = MobileNetV3Large(input_shape=(224,224,3), include_top=False, weights='imagenet')

model_name = 'MobileNetV3L'

In [ ]:
# Freeze the pre-trained layers
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
# Create a Sequential model
x = base_model.output
x = Dropout(0.3)(x)  # Add a dropout layer
x = GlobalAveragePooling2D()(x)

predictions = Dense(len(class_names), activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# Compile the model:
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
import time
from tensorflow.keras.callbacks import EarlyStopping

# Define early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

# Record training start time
training_start_time = time.perf_counter()

# train the model (replace this with your model)
history = model.fit(train_dataset, validation_data=val_dataset, epochs=30, callbacks=[early_stopping])

# Record training end time
training_end_time = time.perf_counter()

# Calculate total training time in minutes
total_training_minutes = (training_end_time - training_start_time) / 60


print(f"\nTotal training time: {total_training_minutes:.2f} minutes")


# Evaluation

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef
)

# 1. Evaluate model
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=1)

# 2. True labels
test_true_labels = np.concatenate([labels.numpy().ravel() for _, labels in test_dataset]).astype(np.int32)

# 3. Predictions
test_pred_probs = model.predict(test_dataset, verbose=1)
test_pred_labels = np.argmax(test_pred_probs, axis=1)

# 4. Metrics
precision = precision_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
recall = recall_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
f1 = f1_score(test_true_labels, test_pred_labels, average="macro", zero_division=0)
cm = confusion_matrix(test_true_labels, test_pred_labels, labels=np.arange(NUM_CLASSES))

# Specificity
specificity_scores = []
for i in range(NUM_CLASSES):
    tp, fn = cm[i, i], cm[i, :].sum() - cm[i, i]
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    specificity_scores.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
avg_specificity = np.mean(specificity_scores)

# Cohen’s Kappa & MCC
kappa = cohen_kappa_score(test_true_labels, test_pred_labels, labels=np.arange(NUM_CLASSES))
mcc = matthews_corrcoef(test_true_labels, test_pred_labels)

# Results
print(f"\n{model_name} Performance Metrics\n" + "-"*55)
print(f"Test Loss: {test_loss:.2f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Macro Precision: {precision*100:.2f}%")
print(f"Macro Recall: {recall*100:.2f}%")
print(f"Macro F1-Score: {f1*100:.2f}%")
print(f"Average Specificity: {avg_specificity*100:.2f}%")
print(f"Cohen's Kappa: {kappa:.2f}")
print(f"Matthews Corrcoef (MCC): {mcc:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], marker='o', label='Training Loss')
plt.plot(history.history['val_loss'], marker='o', label='Validation Loss')

plt.title(f'{model_name} - Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Add grid with black color
plt.grid(True, color='black', linestyle='--', alpha=0.7)

# Ensure borders (spines) are visible and styled
for spine in plt.gca().spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)
    spine.set_color("black")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# Generate confusion matrix
conf_matrix = confusion_matrix(test_true_labels, test_pred_labels)

# Global percentages (counts relative to total samples)
conf_percent = conf_matrix.astype('float') / conf_matrix.sum() * 100

# Annotate each cell with "count\nxx.xx%"
annot_text = np.array([
    [f"{conf_matrix[i, j]}\n{conf_percent[i, j]:.2f}%" for j in range(conf_matrix.shape[1])]
    for i in range(conf_matrix.shape[0])
])

# Plot confusion matrix
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    conf_matrix,
    annot=annot_text,
    fmt='',
    cmap='Blues',
    cbar=True,
    xticklabels=class_names,
    yticklabels=class_names,
    annot_kws={"size": 14, "weight": "bold"},
    linecolor='black',
    linewidths=1
)

# Adjust text color dynamically
for text in ax.texts:
    x, y = text.get_position()
    i, j = int(y - 0.5), int(x - 0.5)
    facecolor = ax.collections[0].get_facecolor()[i * conf_matrix.shape[1] + j]
    r, g, b, _ = facecolor
    brightness = (r + g + b) / 3
    text.set_color("white" if brightness < 0.5 else "black")

# Labels and title
plt.xlabel('Predicted labels', fontsize=12, fontweight="bold")
plt.ylabel('True labels', fontsize=12, fontweight="bold")
plt.title(f'{model_name} - Confusion Matrix', fontsize=16, fontweight="bold")

plt.xticks(rotation=45, ha="right", fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Interactive mode (works in Jupyter/Notebook; in scripts use plt.show(block=False))
plt.ion()

y_true = np.asarray(test_true_labels)
y_score = np.asarray(test_pred_probs)
class_names = list(class_names)
n_classes = len(class_names)
model_name = str(model_name)

y_true_bin = label_binarize(y_true, classes=range(n_classes))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.close("all")
fig, ax = plt.subplots(figsize=(6, 5))

# Light blue background
ax.set_facecolor("aliceblue")

cmap = plt.get_cmap("tab10" if n_classes <= 10 else "tab20")
colors = [cmap(i % cmap.N) for i in range(n_classes)]

# Diagonal reference
ax.plot([0, 1], [0, 1], "--", linewidth=1.3, color="0.4", zorder=1)

# ROC curves
for i, cname in enumerate(class_names):
    ax.plot(fpr[i], tpr[i],
            linewidth=2.2,
            color=colors[i],
            label=f"{cname} (AUC={roc_auc[i]:.3f})",
            zorder=3)

# Axis limits
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Labels
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title(f"{model_name} — ROC Curves",
             fontsize=12, fontweight="bold", pad=10)

# ---- Structured Grid ----
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_xticks(np.arange(0, 1.01, 0.05), minor=True)
ax.set_yticks(np.arange(0, 1.01, 0.05), minor=True)

ax.grid(which="major", linestyle="-", linewidth=0.8, alpha=0.4)
ax.grid(which="minor", linestyle=":", linewidth=0.6, alpha=0.35)
ax.set_axisbelow(True)

# Spines
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(1.0)

# Legend outside, bottom center
legend = ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.25),   # bottom center outside
    frameon=False,
    fontsize=8.5,
    ncol=2
)

plt.tight_layout()
plt.show()


# Model Info

In [ ]:
model.summary()

# XAI

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D

# Find the last convolutional layer in the full model
for layer in reversed(model.layers):  # Start from the end
    if isinstance(layer, Conv2D):  # Check if the layer is a convolutional layer
        print(f"Last Conv Layer: {layer.name}, Output Shape: {layer.output.shape}")
        break

# --- Use Grad-CAM for each class image ---
last_conv_layer_name = layer.name  # replace with printed layer name

# Grad-CAM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image


def gradcam(img_path, model, last_conv_layer_name, size=(224, 224)):
    # Load image
    img = image.load_img(img_path, target_size=size, color_mode="rgb")
    img_array = np.expand_dims(image.img_to_array(img), axis=0)

    # Model returning convolution output and predictions
    grad_model = Model(
        model.inputs,
        [
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    # Calculate gradients
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array, training=False)
        predicted_class = tf.argmax(predictions[0])
        class_score = predictions[:, predicted_class]

    gradients = tape.gradient(class_score, conv_output)
    weights = tf.reduce_mean(gradients, axis=(0, 1, 2))

    # Create heatmap
    heatmap = tf.reduce_sum(conv_output[0] * weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + tf.keras.backend.epsilon()

    # Original image in [0, 1]
    original = image.img_to_array(
        image.load_img(img_path, color_mode="rgb")
    ) / 255.0

    # Resize and color heatmap
    heatmap = tf.image.resize(
        heatmap[..., None],
        original.shape[:2]
    ).numpy().squeeze()

    colored_heatmap = mpl.colormaps["jet"](
        np.clip(heatmap, 0, 1)
    )[..., :3]

    # Blend and clip
    overlay = np.clip(
        0.6 * original + 0.4 * colored_heatmap,
        0,
        1
    )

    return (
        overlay,
        int(predicted_class.numpy()),
        float(tf.reduce_max(predictions[0]).numpy())
    )


# Ensure the grid is large enough
if rows * cols < len(class_images):
    raise ValueError("rows × cols is smaller than the number of class images.")


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows)
)

axes = np.asarray(axes).reshape(-1)


for i, (true_class, img_path) in enumerate(class_images.items()):
    overlay, pred_id, confidence = gradcam(
        img_path,
        model,
        last_conv_layer_name,
        size=(IMG_HEIGHT, IMG_WIDTH)
    )

    axes[i].imshow(overlay)
    axes[i].set_title(
        f"True: {true_class}\n"
        f"Pred: {class_names[pred_id]} ({confidence:.1%})",
        fontsize=10
    )
    axes[i].axis("off")


# Hide unused axes
for i in range(len(class_images), len(axes)):
    axes[i].axis("off")


plt.suptitle(
    f"{model_name} Grad-CAM",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# Lime

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from lime import lime_image
from skimage.segmentation import mark_boundaries
from skimage.segmentation import slic


# =========================================================
# Settings
# =========================================================

input_size = (224, 224)
num_samples = 1000
batch_size = 64
num_features_vis = 8
positive_only = True
hide_color = 0


# =========================================================
# CPU configuration
# =========================================================

LIME_DEVICE = "/CPU:0"

print("LIME inference device:", LIME_DEVICE)


# =========================================================
# Helpers
# =========================================================

def load_rgb(path):
    image_bgr = cv2.imread(path)

    if image_bgr is None:
        raise FileNotFoundError(
            f"Could not load image: {path}"
        )

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    return cv2.resize(
        image_rgb,
        input_size,
        interpolation=cv2.INTER_AREA
    )


def predict_fn(images):
    """
    LIME passes batches of RGB images with values near 0–255.

    All model inference performed for LIME is explicitly
    placed on the CPU.
    """
    images = np.asarray(
        images,
        dtype=np.float32
    )

    with tf.device(LIME_DEVICE):
        images = tf.convert_to_tensor(
            images,
            dtype=tf.float32
        )

        predictions = model(
            images,
            training=False
        )

    return predictions.numpy()


def create_lime_overlay(explainer, rgb_image):
    prediction = predict_fn(
        rgb_image[np.newaxis, ...]
    )[0]

    predicted_index = int(
        np.argmax(prediction)
    )

    explanation = explainer.explain_instance(
        image=rgb_image.astype(np.double),
        classifier_fn=predict_fn,
        top_labels=1,
        labels=(predicted_index,),
        hide_color=hide_color,
        num_samples=num_samples,
        batch_size=batch_size,
        segmentation_fn=lambda image: slic(
            image,
            n_segments=50,
            compactness=10,
            sigma=1,
            start_label=0
        ),
        random_seed=42
    )

    explained_image, mask = explanation.get_image_and_mask(
        label=predicted_index,
        positive_only=positive_only,
        num_features=num_features_vis,
        hide_rest=False
    )

    # mark_boundaries expects a float image in [0, 1].
    explained_image = np.clip(
        explained_image / 255.0,
        0.0,
        1.0
    )

    overlay = mark_boundaries(
        explained_image,
        mask,
        color=(1, 1, 0),
        mode="thick"
    )

    return (
        np.clip(overlay, 0.0, 1.0),
        predicted_index,
        float(prediction[predicted_index])
    )


# =========================================================
# Generate LIME explanations
# =========================================================

explainer = lime_image.LimeImageExplainer(
    random_state=42
)

lime_results = []

for true_class, path in class_images.items():
    print(f"Generating CPU LIME explanation: {true_class}")

    rgb_image = load_rgb(path)

    overlay, predicted_index, confidence = (
        create_lime_overlay(
            explainer,
            rgb_image
        )
    )

    predicted_name = class_names[
        predicted_index
    ]

    lime_results.append({
        "image": overlay,
        "true_class": true_class,
        "predicted_class": predicted_name,
        "confidence": confidence
    })


# =========================================================
# Plot explanations
# =========================================================

if rows * cols < len(lime_results):
    raise ValueError(
        f"The {rows} × {cols} grid has only "
        f"{rows * cols} positions for "
        f"{len(lime_results)} images."
    )


fig, axes = plt.subplots(
    rows,
    cols,
    figsize=(4 * cols, 4 * rows),
    dpi=150
)

axes = np.asarray(axes).reshape(-1)


for index, result in enumerate(lime_results):
    axes[index].imshow(
        result["image"]
    )

    axes[index].set_title(
        f"True: {result['true_class']}\n"
        f"Pred: {result['predicted_class']} "
        f"({result['confidence']:.1%})",
        fontsize=9,
        fontweight="bold"
    )

    axes[index].axis("off")


for index in range(
    len(lime_results),
    len(axes)
):
    axes[index].axis("off")


fig.suptitle(
    f"{model_name} LIME Explanations",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()